# QLoRA Fine-tuning of Qwen2.5 7B on SysPromptRecon Dataset

This notebook fine-tunes Qwen2.5 7B on the SysPromptRecon dataset using Unsloth for efficient QLoRA training.

Dataset: https://huggingface.co/datasets/ruben-blok/SysPromptRecon-V1

Features:
- `input`: question + answer
- `output`: system prompt

Model output will be pushed to: ruben-blok/SysPromptRecon-V1-7B

In [ ]:
# Install required packages
!pip install unsloth[amp]>=2024-1-8 -q
!pip install bitsandbytes>=0.43.0 -q
!pip install peft>=0.10.0 -q
!pip install trl>=0.8.6 -q
!pip install datasets>=2.16.0 -q
!pip install accelerate>=0.27.0 -q
!pip install huggingface_hub>=0.22.0 -q
!pip install git+https://github.com/huggingface/transformers.git -q

# Restart runtime to apply changes
import os
os._exit(0)

In [ ]:
# Import libraries
import torch
from unsloth import FastLanguageModel
from transformers import TrainingArguments, TextStreamer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_int8_training
from trl import SFTTrainer
import os

# Set Hugging Face token (you need to set this in Colab secrets)
# Get your token from https://huggingface.co/settings/tokens
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

# Login to Hugging Face
from huggingface_hub import login
login(token=hf_token)

## Load Dataset

Load the SysPromptRecon dataset from Hugging Face.

In [ ]:
# Load dataset
dataset = load_dataset('ruben-blok/SysPromptRecon-V1')

# Print dataset info
print(dataset)

# Show first example
print('\nFirst training example:')
print(dataset['train'][0])

## Load Model

Load Qwen2.5 7B using Unsloth for fast loading.

In [ ]:
# Model name: Qwen2.5 7B (as per original project)
model_name = "Qwen/Qwen2.5-7B"

# Load model with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,  # Auto-detect
    load_in_4bit = True,  # Use 4-bit quantization for memory efficiency
)

# Print model info
print(f"Model loaded: {model_name}")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

## Configure QLoRA

Set up LoRA adapters for efficient fine-tuning.

In [ ]:
# LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,  # Dropout for LoRA layers
    bias = "none",  # Bias type
    use_gradient_checkpointing = "unsloth",  # Use Unsloth's gradient checkpointing
    random_state = 3407,
    use_rslora = False,  # Use rank-stabilized LoRA
    loftq_config = None,  # LoFTQ configuration
)

# Print trainable parameters
model.print_trainable_parameters()

## Prepare Dataset for Training

Format the dataset for supervised fine-tuning.

In [ ]:
# Formatting function
def formatting_prompts_func(examples):
    # Combine input and output for training
    # Input: question + answer
    # Output: system prompt
    texts = []
    for inp, out in zip(examples['input'], examples['output']):
        # Format: =user
        #         {input}
        #          =assistant
        #         {output}
        text = f"=user\n{inp}=assistant\n{out}=eot"
        texts.append(text)
    return {"text": texts}

# Apply formatting
train_dataset = dataset['train'].map(formatting_prompts_func, batched=True)
eval_dataset = dataset['test'].map(formatting_prompts_func, batched=True)

# Check formatting
print('\nFormatted example:')
print(train_dataset[0]['text'])

## Set Up Training Arguments

Configure training parameters for QLoRA fine-tuning.

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size = 2,  # Batch size per GPU
    gradient_accumulation_steps = 4,  # Gradient accumulation
    warmup_steps = 5,
    max_steps = 100,  # Set to 100 for testing, increase for full training
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",  # Disable wandb and other reporters for simplicity
)

## Initialize Trainer

Set up the SFTTrainer for supervised fine-tuning.

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,  # Can set to True for faster training
    args = training_args,
)

## Start Training

Begin the fine-tuning process.

In [ ]:
# Train the model
trainer.train()

## Save and Push Model

Save the LoRA adapter and push to Hugging Face Hub.

In [ ]:
# Save the adapter locally
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Push to Hugging Face Hub
model.push_to_hub("ruben-blok/SysPromptRecon-V1-7B", token=hf_token)
tokenizer.push_to_hub("ruben-blok/SysPromptRecon-V1-7B", token=hf_token)

print("Model pushed successfully!")

## Inference Example

Test the fine-tuned model with a sample input.

In [ ]:
# Load the fine-tuned model for inference
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "ruben-blok/SysPromptRecon-V1-7B",  # Your Hugging Face repo
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# Example input (question + answer)
input_text = "=user\nWhat is the capital of France?=assistant\n"

# Tokenize
inputs = tokenizer(input_text, return_tensors = "pt").to("cuda")

# Generate
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
response = tokenizer.batch_decode(outputs)[0]

print("Response:")
print(response)